# Lab 9, Part 4: демо без рахунку за Google Cloud

Запускати в Colab з твоїм Google-акаунтом. Потрібен лише BigQuery sandbox: картка не потрібна. Looker Studio — окремо, в браузері (розділ 3).

Перед запуском: **Runtime → Run all** на свіжому runtime має пройти без помилок. Заміни `PROJECT_ID`.

In [ ]:
# 1. Авторизація і ID проєкту.
# Створи sandbox-проєкт без картки (console.cloud.google.com → BigQuery → Enable sandbox)
# або використай свій проєкт з увімкненим BigQuery.
PROJECT_ID = "REPLACE-WITH-YOUR-PROJECT-ID"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print("Connected to", client.project)

## 1. BigQuery: запит до публічного датасету

Спочатку dry run: він показує, скільки байтів запит прочитає. У BigQuery платять саме за байти.

In [ ]:
query = """
SELECT word, SUM(word_count) AS total
FROM `bigquery-public-data.samples.shakespeare`
GROUP BY word
ORDER BY total DESC
LIMIT 10
"""

dry = client.query(query, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
print(f"Запит прочитає {dry.total_bytes_processed / 1e6:.2f} MB")

df_words = client.query(query).to_dataframe()
df_words

## 2. Дані для Looker Studio: CO₂ за Our World in Data

Берємо тільки країни (рядки з ISO-кодом). Рядки «World», «Europe» тощо без ISO-коду відкидаємо: для карти і топ-10 вони зайві.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
co2 = pd.read_csv(url)
cols = ["country", "year", "iso_code", "population", "co2", "co2_per_capita"]
co2 = co2[cols]
countries = co2[co2["iso_code"].notna() & (co2["year"] >= 1950)]
print(countries.shape)
countries.head()

## 2a. Завантаження в BigQuery

Looker Studio читає дані з BigQuery, тому таблиця лежить у BigQuery, а не в CSV.

In [ ]:
DATASET = "peach_lab9"
TABLE = f"{PROJECT_ID}.{DATASET}.owid_co2"

client.create_dataset(DATASET, exists_ok=True)
job = client.load_table_from_dataframe(
    countries, TABLE,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print("Loaded", client.get_table(TABLE).num_rows, "rows into", TABLE)

## 2b. Топ-10 викидів у 2021 (запит для стовпчастої діаграми)

In [ ]:
top10 = client.query(f"""
SELECT country, co2
FROM `{TABLE}`
WHERE year = 2021
ORDER BY co2 DESC
LIMIT 10
""").to_dataframe()
top10

## 3. Looker Studio (у браузері, не в Colab)

1. lookerstudio.google.com → Create → Data source → BigQuery → твій проєкт → `peach_lab9` → `owid_co2`.
2. Додай три графіки: лінійний (`co2` за `year`, фільтр на `World` або на одну країну), географічну карту (`co2_per_capita` за `iso_code`, за 2021), стовпчастий топ-10 (з запиту 2b).
3. Share → Anyone with the link can view. Скопіюй посилання: воно йде в здачу.
4. Зроби скріншот для здачі.

## Прибирання

Таблиці в sandbox видаляються через 60 днів. Коли закінчиш, видали датасет:

```python
client.delete_dataset(DATASET, delete_contents=True, not_found_ok=True)
```